# Resume RAG â€” Experimentation & Analysis

Ingestion, chunking, retrieval accuracy and latency for the resume RAG pipeline (Milestone 2).

In [1]:
import json, time
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from resume_rag import ResumeRAG
from job_matcher import JobMatcher
from vector_store import VectorStore
from metrics import precision_at_k, measure_latency

BASE = Path.cwd().parent
manifest = json.loads((BASE / "dataset_manifest.json").read_text(encoding="utf-8"))
print(f"{len(manifest['resumes'])} resumes across {len({r['role'] for r in manifest['resumes']})} roles, "
      f"{len(manifest['job_descriptions'])} job descriptions")

32 resumes across 8 roles, 5 job descriptions


## 1. Ingest the dataset

In [2]:
store = VectorStore(persist_dir=BASE / "chroma_db_notebook")
rag = ResumeRAG(resumes_dir=BASE / "resumes", vector_store=store)
ingest_summary = rag.ingest()
ingest_summary

C:\Users\Lenovo\Documents\resume-rag-matcher\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21713.58it/s]

{'resumes_processed': 32, 'resumes_failed': 0, 'chunks_created': 160}

## 2. Inspect section-aware chunking on a sample resume

In [3]:
from chunking import chunk_resume
sample_path = next((BASE / "resumes").glob("*.txt"))
chunks = chunk_resume(sample_path.read_text(encoding="utf-8"))
[(c["section"], len(c["text"])) for c in chunks]

[('GENERAL', 62),
 ('SUMMARY', 72),
 ('SKILLS', 47),
 ('EXPERIENCE', 183),
 ('EDUCATION', 38)]

## 3. Retrieval accuracy â€” precision@5 per job description

Ground truth: a resume is "relevant" to a JD if it was generated for that JD's target role (see `dataset_manifest.json`).

In [4]:
matcher = JobMatcher(vector_store=store)
role_by_file = {r["file"]: r["role"] for r in manifest["resumes"]}

# must_have_skills=[] / min_experience_years=None bypass the hard must-have
# filter so this measures the semantic+keyword ranker itself, not the filter
# (with the filter on, only the 1-2 candidates meeting every must-have for
# that role can ever appear, which would make precision@5 near-tautological).
precisions = {}
for jd in manifest["job_descriptions"]:
    jd_text = (BASE / "job_descriptions" / jd["file"]).read_text(encoding="utf-8")
    result = matcher.match(jd_text, top_k=10, must_have_skills=[], min_experience_years=None)
    retrieved_paths = [Path(m["resume_path"]).name for m in result["top_matches"]]
    relevant = {f for f, role in role_by_file.items() if role == jd["role"]}
    precisions[jd["file"]] = precision_at_k(retrieved_paths, relevant, k=5)

precisions

{'jd_ml_engineer.txt': 0.4,
 'jd_data_engineer.txt': 0.4,
 'jd_frontend_developer.txt': 0.4,
 'jd_devops_engineer.txt': 0.4,
 'jd_full_stack_developer.txt': 0.4}

## 4. Latency â€” time per job-matching query

In [5]:
latencies = []
for jd in manifest["job_descriptions"]:
    jd_text = (BASE / "job_descriptions" / jd["file"]).read_text(encoding="utf-8")
    _, elapsed = measure_latency(lambda: matcher.match(jd_text, top_k=10))
    latencies.append(elapsed)

{"mean_latency_s": sum(latencies) / len(latencies), "max_latency_s": max(latencies)}

{'mean_latency_s': 0.02027695999131538, 'max_latency_s': 0.021817600005306304}

## 5. Conclusions

- Section-aware chunking keeps Education/Experience/Skills separable for metadata extraction.
- Hybrid scoring (semantic + keyword) ranks role-matching candidates above off-role ones.
- See the printed precision@5 and latency numbers above for this dataset size.